In [ ]:
# ============================================================
# AUTOENCODER AND VARIATIONAL AUTOENCODER FOR data.npy
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ============================================================
# DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ============================================================
# LOAD DATA
# ============================================================

data = np.load("10_DATA_TRPCAGE.npy")

print("Data shape:", data.shape)

# ============================================================
# NORMALIZATION
# ============================================================

data_mean = data.mean(axis=0)
data_std  = data.std(axis=0) + 1e-8

data_norm = (data - data_mean) / data_std

# ============================================================
# TORCH DATASET
# ============================================================

tensor_data = torch.tensor(data_norm, dtype=torch.float32)

dataset = TensorDataset(tensor_data)

loader = DataLoader(
    dataset,
    batch_size=256,
    shuffle=True
)

# ============================================================
# HYPERPARAMETERS
# ============================================================

input_dim  = data.shape[1]
latent_dim = 2

epochs = 200
lr = 1e-3

# ============================================================
# ============================================================
# AUTOENCODER
# ============================================================
# ============================================================

class AutoEncoder(nn.Module):

    def __init__(self, input_dim, latent_dim):

        super().__init__()

        # ====================================================
        # ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, latent_dim)

        )

        # ====================================================
        # DECODER
        # ====================================================

        self.decoder = nn.Sequential(

            nn.Linear(latent_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, input_dim)

        )

    def forward(self, x):

        z = self.encoder(x)

        xhat = self.decoder(z)

        return xhat, z

# ============================================================
# CREATE MODEL
# ============================================================

AE = AutoEncoder(input_dim, latent_dim).to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(AE.parameters(), lr=lr)

# ============================================================
# TRAIN AUTOENCODER
# ============================================================

ae_losses = []

print("\nTraining AutoEncoder...\n")

for epoch in range(epochs):

    total_loss = 0.0

    for batch in loader:

        x = batch[0].to(device)

        xhat, z = AE(x)

        loss = criterion(xhat, x)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    ae_losses.append(avg_loss)

    print(f"AE Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.6f}")

# ============================================================
# SAVE AE MODEL
# ============================================================

torch.save(AE.state_dict(), "autoencoder.pt")

print("\nAutoEncoder saved.")

# ============================================================
# LATENT REPRESENTATION
# ============================================================

AE.eval()

latent_ae = []

with torch.no_grad():

    for batch in loader:

        x = batch[0].to(device)

        _, z = AE(x)

        latent_ae.append(z.cpu().numpy())

latent_ae = np.concatenate(latent_ae, axis=0)

# ============================================================
# ============================================================
# VARIATIONAL AUTOENCODER
# ============================================================
# ============================================================

class VAE(nn.Module):

    def __init__(self, input_dim, latent_dim):

        super().__init__()

        # ====================================================
        # ENCODER
        # ====================================================

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU()

        )

        # ====================================================
        # LATENT PARAMETERS
        # ====================================================

        self.mu = nn.Linear(64, latent_dim)

        self.logvar = nn.Linear(64, latent_dim)

        # ====================================================
        # DECODER
        # ====================================================

        self.decoder = nn.Sequential(

            nn.Linear(latent_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, input_dim)

        )

    # ========================================================
    # REPARAMETERIZATION
    # ========================================================

    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)

        eps = torch.randn_like(std)

        z = mu + eps * std

        return z

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(self, x):

        h = self.encoder(x)

        mu = self.mu(h)

        logvar = self.logvar(h)

        z = self.reparameterize(mu, logvar)

        xhat = self.decoder(z)

        return xhat, mu, logvar, z

# ============================================================
# CREATE VAE
# ============================================================

VAE_model = VAE(input_dim, latent_dim).to(device)

optimizer_vae = torch.optim.Adam(VAE_model.parameters(), lr=lr)

# ============================================================
# VAE LOSS
# ============================================================

def vae_loss(x, xhat, mu, logvar):

    # ========================================================
    # RECONSTRUCTION LOSS
    # ========================================================

    recon_loss = nn.functional.mse_loss(
        xhat,
        x,
        reduction='mean'
    )

    # ========================================================
    # KL DIVERGENCE
    # ========================================================

    kl_loss = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - torch.exp(logvar)
    )

    total_loss = recon_loss + kl_loss

    return total_loss, recon_loss, kl_loss

# ============================================================
# TRAIN VAE
# ============================================================

vae_total_losses = []
vae_recon_losses = []
vae_kl_losses = []

print("\nTraining VAE...\n")

for epoch in range(epochs):

    total_epoch = 0.0
    recon_epoch = 0.0
    kl_epoch = 0.0

    for batch in loader:

        x = batch[0].to(device)

        xhat, mu, logvar, z = VAE_model(x)

        loss, recon, kl = vae_loss(
            x,
            xhat,
            mu,
            logvar
        )

        optimizer_vae.zero_grad()

        loss.backward()

        optimizer_vae.step()

        total_epoch += loss.item()
        recon_epoch += recon.item()
        kl_epoch += kl.item()

    avg_total = total_epoch / len(loader)
    avg_recon = recon_epoch / len(loader)
    avg_kl = kl_epoch / len(loader)

    vae_total_losses.append(avg_total)
    vae_recon_losses.append(avg_recon)
    vae_kl_losses.append(avg_kl)

    print(
        f"VAE Epoch [{epoch+1}/{epochs}] "
        f"Total: {avg_total:.6f} "
        f"Recon: {avg_recon:.6f} "
        f"KL: {avg_kl:.6f}"
    )

# ============================================================
# SAVE VAE
# ============================================================

torch.save(VAE_model.state_dict(), "vae.pt")

print("\nVAE saved.")

# ============================================================
# LATENT REPRESENTATION OF VAE
# ============================================================

VAE_model.eval()

latent_vae = []

with torch.no_grad():

    for batch in loader:

        x = batch[0].to(device)

        _, mu, _, _ = VAE_model(x)

        latent_vae.append(mu.cpu().numpy())

latent_vae = np.concatenate(latent_vae, axis=0)

# ============================================================
# GENERATE NEW SAMPLES FROM VAE
# ============================================================

with torch.no_grad():

    z = torch.randn(1000, latent_dim).to(device)

    generated = VAE_model.decoder(z).cpu().numpy()

# ============================================================
# DENORMALIZE
# ============================================================

generated = generated * data_std + data_mean

np.save("vae_generated.npy", generated)

print("Generated samples shape:", generated.shape)

# ============================================================
# PLOT AE LATENT SPACE
# ============================================================

plt.figure(figsize=(6,6))

plt.scatter(
    latent_ae[:,0],
    latent_ae[:,1],
    s=5
)

plt.xlabel("Latent Dim 1")
plt.ylabel("Latent Dim 2")

plt.title("AutoEncoder Latent Space")

plt.tight_layout()

plt.savefig("ae_latent.png", dpi=300)

# ============================================================
# PLOT VAE LATENT SPACE
# ============================================================

plt.figure(figsize=(6,6))

plt.scatter(
    latent_vae[:,0],
    latent_vae[:,1],
    s=5
)

plt.xlabel("Latent Dim 1")
plt.ylabel("Latent Dim 2")

plt.title("VAE Latent Space")

plt.tight_layout()

plt.savefig("vae_latent.png", dpi=300)

# ============================================================
# LOSS CURVES
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(ae_losses, label="AE Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("AutoEncoder Loss")

plt.legend()

plt.tight_layout()

plt.savefig("ae_loss.png", dpi=300)

# ============================================================
# VAE LOSSES
# ============================================================

plt.figure(figsize=(8,5))

plt.plot(vae_total_losses, label="Total")

plt.plot(vae_recon_losses, label="Recon")

plt.plot(vae_kl_losses, label="KL")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("VAE Losses")

plt.legend()

plt.tight_layout()

plt.savefig("vae_losses.png", dpi=300)

# ============================================================
# COMPARE REAL VS GENERATED
# ============================================================

plt.figure(figsize=(6,6))

plt.scatter(
    data[:,0],
    data[:,1],
    s=5,
    alpha=0.5,
    label="Real"
)

plt.scatter(
    generated[:,0],
    generated[:,1],
    s=5,
    alpha=0.5,
    label="Generated"
)

plt.xlabel("Dim 1")
plt.ylabel("Dim 2")

plt.title("Real vs Generated")

plt.legend()

plt.tight_layout()

plt.savefig("real_vs_generated.png", dpi=300)

plt.show()

print("\nAll plots saved.")